# Part 3: pi0-FAST — The Tokenization Experiment

## Notebook 10 — pi0-FAST: Autoregressive Action Generation

pi0-FAST replaces pi0's flow matching action head with autoregressive FAST token prediction. The VLM backbone (SigLIP + Gemma 2B) directly outputs discrete action tokens — no separate action expert.


### 1. Load pi0-FAST configuration


In [ ]:
from lerobot.policies.pi0_fast.configuration_pi0_fast import PI0FastConfig

cfg = PI0FastConfig()
print(f"Policy type: pi0-FAST (Autoregressive VLA)")
print(f"Action tokenizer:    {cfg.action_tokenizer_name}")
print(f"Max action tokens:   {cfg.max_action_tokens}")  # 256
print(f"Fast skip tokens:    {cfg.fast_skip_tokens}")  # 128
print(f"Chunk size:          {cfg.chunk_size}")  # 50
print(f"Action steps:        {cfg.n_action_steps}")  # 50
print(f"Action expert:       {cfg.action_expert_variant}")  # gemma_300m


### 2. Architecture: no action expert

Unlike pi0 and pi0.5, pi0-FAST does NOT use a separate action expert. The Gemma 2B backbone directly predicts action tokens. This is the key architectural difference.


In [ ]:
# pi0-FAST Architecture (conceptual)
print("pi0-FAST Architecture:")
print("┌─────────────────────────────────────────┐")
print("│  PaliGemma VLM (SigLIP + Gemma 2B)      │")
print("│  ┌─────────┐  ┌──────────────────┐      │")
print("│  │ SigLIP   │  │ Gemma 2B         │      │")
print("│  │ (Vision) │  │ (Everything!)    │      │")
print("│  └────┬─────┘  └────────┬─────────┘      │")
print("│       └───────┬─────────┘                │")
print("│               ▼                          │")
print("│  ┌──────────────────────────────────┐    │")
print("│  │  Autoregressive Token Decoding    │    │")
print("│  │  Predict FAST token T₁, T₂, ...  │    │")
print("│  │  → decode_actions_with_fast()     │    │")
print("│  │  → Continuous action chunk        │    │")
print("│  └──────────────────────────────────┘    │")
print("└─────────────────────────────────────────┘")

print("\nKey difference from pi0:")
print("  pi0:      VLM + Action Expert (Gemma 300M)")
print("  pi0-FAST: VLM only — no action expert")


### 3. Forward pass: teacher forcing with FAST tokens

During training, pi0-FAST sees the ground-truth FAST tokens (teacher forcing). The loss is standard cross-entropy over the action token vocabulary.


In [ ]:
# Training flow (from modeling_pi0_fast.py):
# 1. Encode action chunk → FAST tokens (pre-computed in dataset)
# 2. Feed [images, text, state, action_tokens[:t]] to model
# 3. Model predicts action_tokens[t+1]
# 4. Cross-entropy loss over token vocabulary

print("pi0-FAST Training:")
print("  Input:  [images, text, state, FAST_tokens[:-1]]")
print("  Target: FAST_tokens")
print("  Loss:   CrossEntropy over FAST token vocabulary")
print("  Same objective as language modeling!")


### 4. Inference: autoregressive token generation

At inference, pi0-FAST generates action tokens one at a time, conditioning on previously generated tokens. KV-caching avoids recomputing the image/text prefix at each step.


In [ ]:
# Inference flow:
# 1. Encode images + text (done once, cached via KV-cache)
# 2. Generate token T₀ (BOS)
# 3. For t = 1 to max_action_tokens:
#       predict T_t from [images, text, T₀..T_{t-1}]
#       if T_t == EOS: break
# 4. Decode tokens → continuous actions via inverse DCT

print("pi0-FAST Inference:")
print("  KV-caching: images/text prefix computed once")
print("  Max decoding steps: 256")
print("  Typical tokens per chunk: 30-60")
print("  Inference speed: ~5 Hz (autoregressive bottleneck)")


### 5. decode_actions_with_fast: tokens → actions

The inverse pipeline: BPE decode → unflatten → inverse DCT → continuous actions.


In [ ]:
# decode_actions_with_fast (from modeling_pi0_fast.py):
# 1. For each generated FAST token:
#     - BPE decode → list of quantized DCT coefficients
# 2. Reshape coefficients → (time_horizon, action_dim) matrix
# 3. Unscale: coeffs / scale
# 4. Inverse DCT along time axis: idct(coeffs, axis=0, norm='ortho')
# 5. Result: continuous action chunk (50 × 7)

print("FAST Decode Pipeline:")
print("  Action tokens → BPE decode → DCT coeffs")
print("  DCT coeffs → unscale → iDCT → action chunk")
print("  Fully invertible (lossless in theory, near-lossless in practice)")


### 6. The inference speed problem

pi0-FAST's major weakness: autoregressive decoding is SLOW. Generating 30-60 tokens sequentially takes much longer than pi0's single-pass flow matching. This is why pi0.5 went back.


In [ ]:
# Speed comparison
print("Inference speed comparison:")
print("  pi0 (flow matching):    ~10-25 Hz (single pass + ODE steps)")
print("  pi0-FAST (tokens):      ~5 Hz    (sequential token decoding)")
print("  pi0.5 (flow matching):  ~10-25 Hz (back to flow matching)")

print("The trade-off:")
print("  pi0-FAST:  5× faster TRAINING, but slower INFERENCE")
print("  pi0/pi0.5: slower training, but faster inference")
print("  For real robots: inference speed matters more")


### The Trade-Off

pi0-FAST achieves 5× faster training by using the same cross-entropy objective as language models. But autoregressive token generation is slow at inference time — a fundamental limitation for real-time robot control. In the final notebook, we compare all approaches side by side.
